# Elastic Net Regression (EN) on MHAR Feature Set

**Purpose:** This notebook implements the Elastic Net (EN) model of Zou and Hastie (2005) on the MHAR feature set for the EURO STOXX 50 index, following the regularized-linear-model framework in Christensen, Siggaard, and Veliyev (2023). Elastic Net combines L1 and L2 regularization, allowing the model to balance variable selection with coefficient shrinkage. In the paper, the mixing parameter and the overall penalty strength are tuned in the validation sample after standardizing the predictors. We mirror that logic here and keep the rest of the forecasting design aligned with the HAR benchmark notebooks so the results are directly comparable.

## 1. Imports

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import ElasticNet


## 2. Data Loading

We keep the same `MALL.csv` data source as in the benchmark and ridge notebooks. This ensures the Elastic Net results are based on the exact same target, predictors, and forecast horizon as the rest of the MHAR model family.

In [ ]:
# Load MALL dataset
MALL = pd.read_csv("/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/MALL.csv")
MALL["Date"] = pd.to_datetime(MALL["Date"])
MALL = MALL.set_index("Date")

print(f"Sample period: {MALL.index[0].date()} to {MALL.index[-1].date()}")
print(f"Number of observations: {len(MALL)}")
display(MALL[["RVD", "RVW", "RVM", "RV_target"]].head())


## 3. Data Split

We again use the 70/10/20 train-validation-test split. The separation matters here because the Elastic Net requires tuning both the overall regularization strength and the mixing parameter between L1 and L2 shrinkage.

In [ ]:
# Train / Validation / Test split (70 / 10 / 20)
n = len(MALL)
n_train = int(0.70 * n)
n_val = int(0.10 * n)
n_test = n - n_train - n_val

print(f"Training observations:   {n_train}")
print(f"Validation observations: {n_val}")
print(f"Test observations:       {n_test}")

print(f"Training period:   {MALL.index[0].date()} to {MALL.index[n_train - 1].date()}")
print(f"Validation period: {MALL.index[n_train].date()} to {MALL.index[n_train + n_val - 1].date()}")
print(f"Test period:       {MALL.index[n_train + n_val].date()} to {MALL.index[-1].date()}")


## 4. Feature Set Construction and Standardization

As stressed in the paper, regularization should be applied to standardized inputs so the penalty treats the regressors on a comparable scale. We therefore standardize the MHAR variables with training-sample moments and keep the target in levels.

In [ ]:
# Define target and MHAR feature set
y = MALL["RV_target"].values
X_MHAR = MALL[["RVD", "RVW", "RVM"]].values

# Split into train, validation and test sets
X_train = X_MHAR[:n_train]
X_val = X_MHAR[n_train:n_train + n_val]
X_test = X_MHAR[n_train + n_val:]

y_train = y[:n_train]
y_val = y[n_train:n_train + n_val]
y_test = y[n_train + n_val:]

# Compute standardization parameters from training set only
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std = np.where(X_std == 0, 1.0, X_std)

# Scale full dataset using training-set parameters
X_scaled = (X_MHAR - X_mean) / X_std

X_train_scaled = X_scaled[:n_train]
X_val_scaled = X_scaled[n_train:n_train + n_val]
X_test_scaled = X_scaled[n_train + n_val:]

print("Feature set: RVD, RVW, RVM")
print(f"\nFull dataset:   X={X_scaled.shape}, y={y.shape}")
print(f"Training set:   X={X_train_scaled.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val_scaled.shape}, y={y_val.shape}")
print(f"Test set:       X={X_test_scaled.shape}, y={y_test.shape}")
print("\nStandardization check on training sample:")
print(f"  Means: {X_train_scaled.mean(axis=0).round(6)}")
print(f"  Std:   {X_train_scaled.std(axis=0).round(6)}")


## 5. Elastic Net Model Specification and Hyperparameter Tuning

The paper tunes both the overall penalty and the mixing parameter in the validation set. We therefore evaluate a broad logarithmic alpha grid together with 10 values for the Elastic Net mixing parameter, spanning ridge-like to lasso-like penalization.

In [ ]:
# Hyperparameter grids following the paper
alpha_grid = np.logspace(-5, 2, 1000)
l1_ratio_grid = np.linspace(0.1, 1.0, 10)

tuning_records = []
best_model = None
best_mse = np.inf
best_alpha = None
best_l1_ratio = None

for l1_ratio in l1_ratio_grid:
    for alpha in alpha_grid:
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, fit_intercept=True, max_iter=100000)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
        mse = np.mean((y_val - y_pred) ** 2)
        tuning_records.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "val_mse": mse,
            "nonzero": np.sum(np.abs(model.coef_) > 1e-12)
        })
        if mse < best_mse:
            best_mse = mse
            best_alpha = alpha
            best_l1_ratio = l1_ratio
            best_model = model

tuning_results = pd.DataFrame(tuning_records)
heatmap = tuning_results.pivot(index="l1_ratio", columns="alpha", values="val_mse")
nonzero_heatmap = tuning_results.pivot(index="l1_ratio", columns="alpha", values="nonzero")

print(f"Alpha range: {alpha_grid[0]:.2e} to {alpha_grid[-1]:.2e}")
print(f"l1_ratio grid: {np.round(l1_ratio_grid, 2)}")
print(f"Optimal alpha: {best_alpha:.6e}")
print(f"Optimal l1_ratio: {best_l1_ratio:.2f}")
print(f"Validation MSE: {best_mse:.6e}")
print(f"Non-zero coefficients at optimum: {np.sum(np.abs(best_model.coef_) > 1e-12)}")

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(heatmap.values, aspect="auto", origin="lower", cmap="viridis")
ax.set_title("Elastic Net — Validation MSE Across Hyperparameter Grid\nMHAR Feature Set", fontsize=11)
ax.set_xlabel("Alpha grid index (low to high regularization)")
ax.set_ylabel("l1_ratio")
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels([f"{v:.2f}" for v in heatmap.index])
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Validation MSE")
plt.tight_layout()
plt.show()


## 6. Rolling Window Estimation and Forecasting

We now carry the best validation-set hyperparameters into the rolling forecast exercise. Each forecast is formed one day ahead using a window of length equal to the initial training plus validation sample, matching the benchmark design used throughout the thesis notebooks.

In [ ]:
fixed_alpha = best_alpha
fixed_l1_ratio = best_l1_ratio
window_size = n_train + n_val

print(f"Fixed alpha: {fixed_alpha:.6e}")
print(f"Fixed l1_ratio: {fixed_l1_ratio:.2f}")
print(f"Rolling window length: {window_size}")

forecasts_EN = []
dates_test = []
nonzero_path = []

start_time = time.time()

for i in range(n_test):
    X_window = X_scaled[i:i + window_size]
    y_window = y[i:i + window_size]

    model_final = ElasticNet(
        alpha=fixed_alpha,
        l1_ratio=fixed_l1_ratio,
        fit_intercept=True,
        max_iter=100000
    )
    model_final.fit(X_window, y_window)

    X_forecast = X_scaled[i + window_size].reshape(1, -1)
    forecast = model_final.predict(X_forecast)[0]
    forecast = max(forecast, y_window.min())

    forecasts_EN.append(forecast)
    dates_test.append(MALL.index[i + window_size])
    nonzero_path.append(np.sum(np.abs(model_final.coef_) > 1e-12))

    if (i + 1) % 100 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"  Day {i + 1:4d}/{n_test} | Date: {MALL.index[i + window_size].date()} | Elapsed: {elapsed:.1f} min")

total_elapsed = (time.time() - start_time) / 60
print(f"\nTotal estimation time: {total_elapsed:.1f} minutes")
print(f"Forecasts generated: {len(forecasts_EN)}")
print(f"Forecast period: {dates_test[0].date()} to {dates_test[-1].date()}")
print(f"Average number of active predictors: {np.mean(nonzero_path):.2f}")


## 7. Save Forecasts

The forecasts and summary metrics are stored using the same file structure as the benchmark notebooks so they can be reused in downstream model-comparison work.

In [ ]:
forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/Forcasts/MHAR"
os.makedirs(forecast_path, exist_ok=True)

test_dates = MALL.index[n_train + n_val:]
results_EN = pd.DataFrame({
    "RV_actual": y_test,
    "RV_forecast": forecasts_EN
}, index=test_dates)

errors_EN = results_EN["RV_actual"] - results_EN["RV_forecast"]
mse_EN = np.mean(errors_EN ** 2)

results_EN.to_csv(os.path.join(forecast_path, "forecasts_EN.csv"))

mse_summary_EN = pd.DataFrame({
    "Model": ["EN"],
    "MSE": [mse_EN],
    "RMSE": [np.sqrt(mse_EN)],
    "MAE": [errors_EN.abs().mean()],
    "Mean_Error": [errors_EN.mean()],
}).set_index("Model")

mse_summary_EN.to_csv(os.path.join(forecast_path, "mse_summary_EN.csv"))

print(f"Forecasts saved to: {forecast_path}")
print(f"Rows saved: {len(results_EN)}")
print(f"Elastic Net test-set MSE: {mse_EN:.6e}")


## 8. Out-of-Sample Performance Evaluation

We compare Elastic Net with the HAR benchmark on the common test set and report the same level-scale accuracy measures as in the benchmark model notebooks.

In [ ]:
forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/Forcasts/MHAR"
results_HAR = pd.read_csv(
    os.path.join(forecast_path, "forecasts_HAR.csv"),
    index_col=0,
    parse_dates=True
)

mse_HAR = np.mean((results_HAR["RV_actual"] - results_HAR["RV_forecast"]) ** 2)
errors_HAR = results_HAR["RV_actual"] - results_HAR["RV_forecast"]
rel_mse_EN = mse_EN / mse_HAR

print(f"Out-of-sample performance ({test_dates[0].date()} to {test_dates[-1].date()}):")
print(f"\n{'Model':<10} {'MSE':>12} {'RMSE':>12} {'MAE':>12} {'Mean Error':>12} {'Rel. MSE':>10}")
print("-" * 70)
print(f"{'HAR':<10} {mse_HAR:>12.6e} {np.sqrt(mse_HAR):>12.6e} {errors_HAR.abs().mean():>12.6e} {errors_HAR.mean():>12.6e} {1.0:>10.4f}")
print(f"{'EN':<10} {mse_EN:>12.6e} {np.sqrt(mse_EN):>12.6e} {errors_EN.abs().mean():>12.6e} {errors_EN.mean():>12.6e} {rel_mse_EN:>10.4f}")
print("-" * 70)
print(f"\nEN vs HAR: {'outperforms' if rel_mse_EN < 1 else 'underperforms'} by {abs(1 - rel_mse_EN) * 100:.2f}%")


## 9. Forecast Visualization

As in the other benchmark notebooks, we present the forecast path in annualized volatility terms to make the scale intuitive and directly comparable across models.

In [ ]:
results_plot_EN = pd.DataFrame({
    "Actual": np.sqrt(results_EN["RV_actual"] * 252) * 100,
    "Forecast": np.sqrt(results_EN["RV_forecast"] * 252) * 100
}, index=results_EN.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_plot_EN["Actual"], label="Actual RV", color="black", linewidth=0.8)
ax.plot(results_plot_EN["Forecast"], label="Elastic Net Forecast", color="red", linewidth=0.8, linestyle="--")
ax.set_title("Elastic Net Regression — One-Day-Ahead Realized Volatility Forecast\nEURO STOXX 50 (2016–2020)", fontsize=12)
ax.set_ylabel("Annualized Volatility (%)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSummary of forecast errors:")
print(f"Mean error:  {errors_EN.mean():.6e}")
print(f"RMSE:        {np.sqrt(mse_EN):.6e}")
print(f"MAE:         {errors_EN.abs().mean():.6e}")
print(f"Average active coefficients in rolling estimation: {np.mean(nonzero_path):.2f}")
